# Mastcam dust opacity (tau) per sol

Turns Mark Lemmon's MSL Mastcam aerosol optical depth tables into `mastcam_tau_per_sol.csv` (columns `sol`, `tau`, `n`, `tau_min`, `tau_max`, `ls_deg`, `band`), one row per sol with a measurement. Used by `../dust_devil/dust_devil_likelihood_demo.ipynb`, and by the temperature model's training table.

**Source:** unzip "Mars Science Laboratory aerosol optical depths" into `raw_data/`. It holds `msl-tau-440-arch*.txt` and `msl-tau-880-arch*.txt` in three versions. Use the **latest** (`-3953`, through sol 3953); the plain `arch` and `-3778` files are older subsets with identical values for the rows they share.

**Things to know**
- Mastcam gives a column optical depth from direct solar images. The demo applies it to broadband sunlight with `exp(-tau/mu)`, which is a simplification. 440 nm values run about 5% below 880 nm on average, so the band choice moves tau only a little. 880 nm is the default.
- Measurements start at sol 33; sols 1-32 have none.
- The 2018 global dust storm reaches tau 8.6 (sols 2082-2133). Some rows there have product IDs ending `DXX5` or `DXX7`; they are kept (a parser that only accepts `DXXX` drops them and silently caps tau near 2).
- One row in the 440 nm files has an unreadable uncertainty (`******`); the uncertainty is read as missing, the tau is kept.
- Several measurements on one sol are reduced to their median. Measurements are made around local noon, so one value per sol is an approximation for the whole day.

Needs: numpy, pandas. No try/except: a changed file format or a row-count mismatch stops the run.

In [1]:
from pathlib import Path
import re
import numpy as np, pandas as pd

RAW = Path("../raw_data")
PROC = Path("../processed_data")
FILES = {"880": RAW / "msl-tau-880-arch-3953.txt", "440": RAW / "msl-tau-440-arch-3953.txt"}   # the two Mastcam tables, used directly
TAU_BAND = "880"          # "880", "440", or "mean" (average of the two bands, sols with both)
REMS_CSV = PROC / "rems_binned_all_sols.csv"     # only used for the coverage report
OUT = PROC / "mastcam_tau_per_sol.csv"
assert TAU_BAND in ("880", "440", "mean"), TAU_BAND

In [2]:
def read_tau(path):
    text = path.read_text()
    declared = int(re.search(r"Lines of data in this table:\s*(\d+)", text).group(1))
    lines = text.splitlines()
    start = next(i for i, l in enumerate(lines) if "Product_ID" in l) + 1
    rows = [l.split(",") for l in lines[start:] if l.strip()]
    assert all(len(r) == 5 for r in rows), f"{path.name}: a row does not have 5 fields"
    d = pd.DataFrame({"sol": [float(r[1]) for r in rows], "ls_deg": [float(r[2]) for r in rows], "tau": [float(r[3]) for r in rows],
                      "sigma": pd.to_numeric([r[4].strip() for r in rows], errors="coerce")})     # '******' -> NaN
    assert len(d) == declared, f"{path.name}: parsed {len(d)} rows but the header declares {declared}"
    n_bad = int((d.tau < 0).sum())                       # -1 = no validated data (none in the current files)
    d = d[d.tau >= 0]
    print(f"{path.name}: {len(d)} rows, sol {d.sol.min():.1f}..{d.sol.max():.1f}, tau {d.tau.min():.3f}..{d.tau.max():.3f}, dropped {n_bad} with tau = -1, {int(d.sigma.isna().sum())} with unreadable sigma")
    return d

def per_sol(d):
    d = d.assign(sol_int=np.floor(d.sol).astype(int))
    return d.groupby("sol_int").agg(tau=("tau", "median"), n=("tau", "size"), tau_min=("tau", "min"), tau_max=("tau", "max"), ls_deg=("ls_deg", "first"))

t880, t440 = per_sol(read_tau(FILES["880"])), per_sol(read_tau(FILES["440"]))
if TAU_BAND == "880":
    out = t880
elif TAU_BAND == "440":
    out = t440
else:
    both = t880.join(t440, lsuffix="_880", rsuffix="_440", how="inner")
    out = pd.DataFrame({"tau": (both.tau_880 + both.tau_440) / 2, "n": both.n_880 + both.n_440,
                        "tau_min": both[["tau_min_880", "tau_min_440"]].min(axis=1), "tau_max": both[["tau_max_880", "tau_max_440"]].max(axis=1),
                        "ls_deg": both.ls_deg_880})
out = out.rename_axis("sol").reset_index()
out["band"] = TAU_BAND
print(f"\n{len(out)} sols with tau ({TAU_BAND}); median tau {out.tau.median():.2f}, 95th percentile {out.tau.quantile(.95):.2f}, max {out.tau.max():.2f} (sol {int(out.sol[out.tau.idxmax()])})")

msl-tau-880-arch-3953.txt: 1938 rows, sol 33.5..3953.5, tau 0.332..8.589, dropped 0 with tau = -1, 0 with unreadable sigma
msl-tau-440-arch-3953.txt: 1903 rows, sol 33.5..3953.5, tau 0.303..7.935, dropped 0 with tau = -1, 1 with unreadable sigma

1358 sols with tau (880); median tau 0.78, 95th percentile 1.61, max 8.59 (sol 2085)


In [3]:
# ---- coverage against the REMS sols you have ----
rems_sols = np.sort(pd.read_csv(REMS_CSV).sol.unique())
dist = np.array([np.abs(out.sol.to_numpy() - s).min() for s in rems_sols])
print(f"REMS sols: {len(rems_sols)} | tau on the same sol: {int((dist == 0).sum())} | within 5 sols: {int((dist <= 5).sum())} | within 10: {int((dist <= 10).sum())} | before the first tau sol ({int(out.sol.min())}): {int((rems_sols < out.sol.min()).sum())}")
gaps = np.diff(out.sol.to_numpy()); print(f"gap between measured sols: median {np.median(gaps):.0f}, 90th percentile {np.percentile(gaps, 90):.0f}, max {gaps.max()} (after sol {int(out.sol.to_numpy()[gaps.argmax()])})")
for s in (800, 1163):
    near = out.iloc[np.abs(out.sol - s).argmin()]; print(f"sol {s}: nearest measured sol {int(near.sol)}, tau {near.tau:.3f} (n={int(near.n)}), Ls {near.ls_deg:.1f}")
out.to_csv(OUT, index=False, float_format="%.4f")
print("wrote", OUT)
out.head()

REMS sols: 272 | tau on the same sol: 65 | within 5 sols: 248 | within 10: 256 | before the first tau sol (33): 25
gap between measured sols: median 2, 90th percentile 5, max 37 (after sol 2174)
sol 800: nearest measured sol 799, tau 1.517 (n=1), Ls 227.5
sol 1163: nearest measured sol 1163, tau 0.524 (n=1), Ls 68.1
wrote processed_data\mastcam_tau_per_sol.csv


,sol,tau,n,tau_min,tau_max,ls_deg,band
0,33,0.7080,2,0.700,0.716,168.5,880
1,37,0.7470,2,0.742,0.752,170.9,880
2,42,0.7020,8,0.684,0.706,173.5,880
3,44,0.7180,1,0.718,0.718,174.6,880
4,48,0.8115,2,0.796,0.827,177.0,880
